# 01 - Exploratory data analysis

Run `python -m src.pipeline --stages data features` first. This notebook only reads pipeline outputs; nothing in the production path lives here.

**Check the data-source line below: if it says SYNTHETIC the findings are about the stand-in data, not UCI Online Retail II.**

In [1]:
import sys, json, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.utils.config import load_config, path_of, artifacts_path
from src.pipeline import read_clean
cfg = load_config()
meta = json.loads(artifacts_path(cfg, "metadata.json").read_text())
print("DATA SOURCE:", meta["data_source"])
sns.set_theme(style="whitegrid")

I0000 00:00:1791043892.253998     165 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


DATA SOURCE: SYNTHETIC STAND-IN (not real data)


I0000 00:00:1791043893.700292     165 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
tx = read_clean(cfg)
print(tx.shape)
tx.head()

(644727, 18)


,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,has_customer_id,is_return,is_price_adjustment,is_admin_code,is_outlier,is_entry_error,is_bulk_outlier,category,line_total,use_for_customer_model
0,903188,21010,SPOTTY SPOON 1,2,2009-12-01 08:00:00.093689960,1.57,<NA>,United Kingdom,False,False,False,False,False,False,False,kitchen_dining,3.14,False
1,900535,28002,PINK TOWEL 1,1,2009-12-01 08:02:13.531555964,1.20,<NA>,United Kingdom,False,False,False,False,False,False,False,bath_beauty,1.20,False
2,489445,23000,VINTAGE CHRISTMAS DECORATION 1,6,2009-12-01 08:03:02.156769140,5.18,14120,United Kingdom,True,False,False,False,False,False,False,christmas_seasonal,31.08,True
3,907993,22036,VINTAGE BAG 4,1,2009-12-01 08:15:35.282892088,1.53,<NA>,United Kingdom,False,False,False,False,False,False,False,bags_storage,1.53,False
4,489442,30004,RED WIDGET 1,12,2009-12-01 08:20:07.953570769,1.42,13445,United Kingdom,True,False,False,False,False,False,False,other,17.04,True


## 1. Data quality overview

In [3]:
rep = json.loads(path_of(cfg, 'cleaning_report').read_text())
pd.Series({k: v for k, v in rep.items() if not isinstance(v, dict)}).to_frame('value')

,value
cleaning_version,1.0.0
rows_in,646649
exact_duplicates_removed,1922
rows_missing_customer_id,115919
rows_out,644727
return_lines,14601
price_adjustment_lines,1281
admin_code_lines,2563
rows_used_for_customer_model,528802
unique_customers,6000


In [4]:
flags = ['has_customer_id','is_return','is_price_adjustment','is_admin_code','is_entry_error','is_bulk_outlier']
(tx[flags].mean()*100).round(2).rename('% of rows').to_frame()

,% of rows
has_customer_id,82.02
is_return,2.26
is_price_adjustment,0.20
is_admin_code,0.40
is_entry_error,0.00
is_bulk_outlier,4.55


## 2. Revenue over time and seasonality

In [5]:
m = tx[tx.use_for_customer_model].set_index('invoice_date')['line_total'].resample('MS').sum()
fig, ax = plt.subplots(figsize=(9,3.5)); ax.plot(m.index, m.values, marker='o'); ax.set_title('Monthly net revenue (customer-attributed)'); plt.show()

In [6]:
d = tx[tx.use_for_customer_model & ~tx.is_return]
by_month = d.groupby(d.invoice_date.dt.month)['invoice'].nunique()
fig, ax = plt.subplots(figsize=(8,3)); ax.bar(by_month.index, by_month.values); ax.set_title('Orders by calendar month (all years)'); ax.set_xlabel('month'); plt.show()

## 3. Customers, countries, categories

In [7]:
print(d.customer_id.nunique(), 'customers')
rc = d.groupby('country')['line_total'].sum().sort_values(ascending=False).head(8)[::-1]
fig, ax = plt.subplots(figsize=(7,3)); ax.barh(rc.index.astype(str), rc.values); ax.set_title('Revenue by country'); plt.show()

6000 customers


In [8]:
rcat = d.groupby('category')['line_total'].sum().sort_values()
fig, ax = plt.subplots(figsize=(7,3.5)); ax.barh(rcat.index.astype(str), rcat.values); ax.set_title('Revenue by keyword-derived category'); plt.show()

## 4. Returns / cancellations

In [9]:
r = tx[tx.use_for_customer_model]
print('return lines %:', round(100*r.is_return.mean(),2))
rv = r[r.is_return].set_index('invoice_date')['line_total'].resample('MS').sum()
fig, ax = plt.subplots(figsize=(9,3)); ax.plot(rv.index, rv.values, marker='o'); ax.set_title('Monthly returned value'); plt.show()

return lines %: 2.52


## 5. Customer-level distributions (training snapshot)

In [10]:
f = pd.read_csv(path_of(cfg, 'features_train'))
fig, axs = plt.subplots(1,4, figsize=(14,3))
for ax, c, lg in zip(axs, ['recency_days','frequency','total_spend','avg_basket_size'], [False, True, True, True]):
    sns.histplot(np.log1p(f[c]) if lg else f[c], bins=40, ax=ax); ax.set_title(('log1p ' if lg else '') + c)
plt.tight_layout(); plt.show()
print('churn rate (no purchase in next 90d):', round(f.churn.mean(),3))

churn rate (no purchase in next 90d): 0.605


In [11]:
num = f.select_dtypes('number').drop(columns=['customer_id'])
plt.figure(figsize=(9,6)); sns.heatmap(num[['recency_days','frequency','total_spend','avg_spend','historical_clv','avg_basket_size','freq_trend','gap_variance','seasonal_concentration','return_rate','discount_sensitivity','engagement_score','churn','clv_target']].corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size':6}); plt.show()

In [12]:
q = f.groupby(pd.qcut(f.recency_days, 5, duplicates='drop'), observed=True)['churn'].mean()
fig, ax = plt.subplots(figsize=(6,3)); ax.bar([str(i) for i in q.index], q.values); ax.set_title('Churn rate by recency quintile'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 6. Hypotheses to test in modelling

Written as testable statements; each is checked in `02_feature_engineering` and `03_model_experiments` (SHAP global importance, partial dependence).

1. **Recency dominates.** Days since last purchase is the single strongest churn predictor; churn probability rises monotonically with recency (confirm with the recency-quintile chart and SHAP).
2. **Frequency protects.** Customers with more orders are less likely to churn, but the effect saturates after a handful of orders.
3. **Declining engagement precedes churn.** A negative `freq_trend` and a long time since the last order relative to the customer's usual gap increase risk beyond what recency alone explains.
4. **Seasonal shoppers look like churners off-season.** High `seasonal_concentration` customers will be over-flagged outside Q4, so a single 90-day cutoff may mislead for this group (check errors by segment).
5. **Returns signal dissatisfaction.** A higher `return_rate` is associated with higher churn, but is confounded with spend (large buyers return more in absolute terms).
6. **Value is heavy-tailed.** A small share of customers generates most revenue; CLV regression will be dominated by outliers, so R² will be sensitive to a few accounts and robust metrics (MAE) must be read alongside it.
7. **Missing Customer ID is not random.** Guest/unattributed lines differ in basket and country mix; excluding them biases customer-level results toward registered buyers.
8. **Trees beat deep nets at this sample size.** With a few thousand customers, gradient-boosted trees should match or beat the ANN/LSTM while being cheaper and explainable.